# 02 · Baseline — Blob Detection + Hungarian Linking

**Bu bir Code Competition.** Görünen `test/` bir *placeholder* (train'den kopyalanmış 4 film);
submit edince Kaggle **gerçek gizli test setini** bağlayıp bu defteri yeniden çalıştırır.
→ Dataset isimleri/sayısı **hardcode edilmez**, `test/` dinamik gezilir.

## Neden Ultrack değil de bu?
EDA (`01_eda`, `01b_eda_detailed`) şunları gösterdi:
- Hareket **< 8 µm** (95p 5.5 / 99p 8) → basit en-yakın-komşu linking çoğu kenarı doğru bağlar
- Kare-içi GT komşuları ~25 µm uzakta → **eşleşme belirsizliği yok**
- Zamansal **boşluk yok** → gap-closing gerekmez
- Çekirdekler arka plandan ~8× parlak → detection kolay

Yani %90'lık edge skorunun büyük kısmı **iyi detection + basit linking** ile erişilebilir.
`tracksdata`/`ultrack` kurulumu Kaggle'da numpy ABI'sini kırıyordu → **bağımlılık-hafif**,
kesin çalışan bir hat kuruyoruz. Ultrack sonraki upgrade.

## Parametreler (EDA'dan)
| Param | Değer | Kaynak |
|---|---|---|
| ölçek (Z,Y,X) µm/px | (1.625, 0.40625, 0.40625) | multiscales |
| linking yarıçapı | **8 µm** | D04 (99p) |
| metrik eşleşme | 7 µm | metrics.md |
| hedef yoğunluk | ~213 çekirdek/kare | D09 |

## 0 · Kurulum — ⚠️ internet KAPALI olmalı

Bu yarışmada submit için **internet kapalı** olmak zorunda, ama `zarr` Kaggle base imajında **yok**.
Tek seferlik çözüm — **zarr wheel'lerini bir Kaggle Dataset'ine koy**:

1. **Yeni bir notebook** aç, **internet AÇIK**:
   ```
   !pip download zarr -d /kaggle/working/wheels
   ```
2. **Save & Run All** → sağdaki *Output*'tan **"New Dataset"** ile dataset oluştur (ör. adı `zarr-wheels`).
3. **Bu notebook'ta:** *Add Data* → `zarr-wheels` dataset'ini ekle.
4. *Settings* → **Internet: Off** → *Save & Run All* → Submit.

Aşağıdaki hücre wheel klasörünü `/kaggle/input` altında **otomatik bulur**; bulamazsa
(editörde, internet açıkken) online kurmayı dener.

In [ ]:
# Bu yarismada SUBMIT icin internet KAPALI olmali; ama zarr base imajda YOK.
# Cozum: zarr wheel'leri bir Kaggle Dataset'inden offline kurulur.
# (Wheel dataset'i nasil hazirlanir -> asagidaki markdown'a bak.)
import sys, subprocess, glob

def ensure_zarr():
    try:
        import zarr; return zarr
    except ImportError:
        pass
    # 1) OFFLINE: /kaggle/input altindaki wheel klasorlerini dene (internet kapali senaryo)
    cands=[]
    for pat in ("/kaggle/input/*/wheels","/kaggle/input/*wheel*","/kaggle/input/*zarr*","/kaggle/input/*"):
        cands+= [d for d in glob.glob(pat) if glob.glob(d+"/*.whl")]
    for d in dict.fromkeys(cands):
        subprocess.run([sys.executable,"-m","pip","install","-q","--no-index",
                        f"--find-links={d}","zarr"], capture_output=True)
        try:
            import zarr; print("zarr OFFLINE kuruldu:", d); return zarr
        except ImportError:
            continue
    # 2) ONLINE: editorde internet acikken (submit oncesi gelistirme)
    print("[uyari] wheel bulunamadi -> internetten kurmayi deniyorum (SUBMIT'te calismaz!)")
    subprocess.run([sys.executable,"-m","pip","install","-q","zarr"], check=False)
    import zarr; return zarr

zarr = ensure_zarr()
print("zarr:", zarr.__version__)

import time, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.filters import threshold_otsu
warnings.filterwarnings("ignore")
print("hazir")

## 1 · Yapılandırma

In [ ]:
SCALE_ZYX = (1.625, 0.40625, 0.40625)   # um/piksel (Z,Y,X)
S = np.array(SCALE_ZYX, dtype=np.float32)

LINK_MAX_UM = 8.0      # linking arama yaricapi (EDA: 99p ~8um)
MATCH_UM    = 7.0      # metrik eslesme toleransi

SIGMA = (1, 2, 2)      # Gauss yumusatma (voxel; Z ince)
FOOT  = (3, 11, 11)    # yerel-maksimum footprint ~ cekirdek boyutu

MAX_TEST = None        # None = tum test datasetleri (gercek rerun icin SART)
OUT_CSV  = "/kaggle/working/submission.csv"

INPUT = Path("/kaggle/input")
def find_root():
    st=[(INPUT,0)]
    while st:
        b,d=st.pop()
        try:
            if (b/"train").is_dir() and (b/"test").is_dir(): return b
        except Exception: pass
        if d<4:
            for c in sorted(b.iterdir()):
                if c.is_dir() and not c.name.endswith((".zarr",".geff")): st.append((c,d+1))
ROOT=find_root(); TRAIN=ROOT/"train"; TEST=ROOT/"test"
train_names=sorted(p.stem for p in TRAIN.glob("*.zarr"))
test_names =sorted(p.stem for p in TEST.glob("*.zarr"))
print("ROOT:",ROOT)
print(f"train={len(train_names)} | test={len(test_names)}")
print("test:",test_names[:10], "..." if len(test_names)>10 else "")

## 2 · Yardımcı okuyucular

In [ ]:
def open_image(zpath):
    n=zarr.open(str(zpath),mode="r"); a=dict(n.attrs)
    ms=a.get("multiscales")
    if ms is None and isinstance(a.get("ome"),dict): ms=a["ome"].get("multiscales")
    if ms:
        return n[ms[0]["datasets"][0]["path"]]
    keys=list(n.keys()) if hasattr(n,"keys") else []
    return n["0"] if "0" in keys else n

def load_geff(gp):
    g=zarr.open(str(gp),mode="r")
    nodes=g["nodes"]; ids=np.asarray(nodes["ids"]); props={}
    if "props" in nodes:
        for pn in list(nodes["props"].keys()):
            try: props[pn]=np.asarray(nodes["props"][pn]["values"])
            except Exception: pass
    edges=np.asarray(g["edges"]["ids"])
    d={"id":ids}
    for k in ("t","z","y","x"):
        if k in props: d[k]=props[k]
    return pd.DataFrame(d), edges
print("ok")

## 3 · Detection — çekirdek merkezleri
Gauss yumuşatma → Otsu eşiği → çekirdek boyutlu yerel-maksimum → bağlı bileşen ağırlık merkezi.
(EDA'da bu hat ~213 çekirdek/kare veriyordu = gerçek yoğunluk.)

In [ ]:
def detect_frame(v):
    sm=ndi.gaussian_filter(v.astype(np.float32), sigma=SIGMA)
    thr=threshold_otsu(sm)
    mx=ndi.maximum_filter(sm, size=FOOT)
    peaks=(sm==mx)&(sm>thr)
    lbl,n=ndi.label(peaks)
    if n==0: return np.zeros((0,3),np.float32)
    c=ndi.center_of_mass(sm, lbl, np.arange(1,n+1))
    return np.asarray(c, dtype=np.float32)     # (N,3) voxel (z,y,x)

# hiz testi + yogunluk kontrolu
_arr=open_image(TEST/(test_names[0]+".zarr"))
t0=time.time(); _c=detect_frame(np.asarray(_arr[50])); t1=time.time()
print(f"1 kare detection: {t1-t0:.2f}s | bulunan cekirdek: {len(_c)}")
print(f"tahmini 1 dataset (100 kare): {(t1-t0)*100:.0f}s")
print(f"tahmini {len(test_names)} test datasi: {(t1-t0)*100*len(test_names)/60:.1f} dk")

## 4 · Linking — Hungarian (8 µm kapılı)
Ardışık karelerde optimal 1-1 eşleştirme; 8 µm üstü bağlantılar yasak.
Eşleşmeyen node = yeni track (beliriş) / track sonu (kayboluş) — EDA'da ikisi de gerçek.

In [ ]:
def link_pairs(A,B):
    if len(A)==0 or len(B)==0: return []
    D=cdist(A*S, B*S)                       # um cinsinden mesafe
    cost=np.where(D<=LINK_MAX_UM, D, 1e6)   # kapi
    r,c=linear_sum_assignment(cost)
    return [(int(i),int(j)) for i,j in zip(r,c) if D[i,j]<=LINK_MAX_UM]

def track_dataset(arr, log=False):
    T=arr.shape[0]; cents=[]
    for t in range(T):
        cents.append(detect_frame(np.asarray(arr[t])))
    nodes=[]; edges=[]; off=[]; nid=1
    for t,c in enumerate(cents):
        off.append(nid)
        for p in c:
            # SEMA: koordinatlar TAMSAYI voxel olmali -> yuvarla (7um tolerans yaninda ihmal edilebilir)
            nodes.append((nid, t, int(round(p[0])), int(round(p[1])), int(round(p[2])))); nid+=1
    for t in range(T-1):
        for i,j in link_pairs(cents[t],cents[t+1]):
            edges.append((off[t]+i, off[t+1]+j))
    if log: print(f"  node={len(nodes)} edge={len(edges)} (~{len(nodes)/T:.0f}/kare)")
    return nodes, edges
print("ok")

## 5 · Yerel metrik (Edge Jaccard yaklaşımı)
Resmi metriğe yakın: node'lar 7 µm ile GT'ye eşleştirilir, sonra kenarlar karşılaştırılır.
**Seyrek GT mantığı:** her iki ucu da GT'ye eşleşmeyen tahmin kenarı **yok sayılır**
(etiketsiz hücre = hata değil).

In [ ]:
def eval_vs_gt(nodes, edges, gt_ndf, gt_edges):
    pn=pd.DataFrame(nodes, columns=["node_id","t","z","y","x"])
    gmap={}
    for t,g in gt_ndf.groupby("t"):
        p=pn[pn.t==int(t)]
        if len(p)==0 or len(g)==0: continue
        D=cdist(p[["z","y","x"]].values*S, g[["z","y","x"]].values*S)
        cost=np.where(D<=MATCH_UM, D, 1e6)
        r,c=linear_sum_assignment(cost)
        pid=p["node_id"].values; gid=g["id"].values
        for i,j in zip(r,c):
            if D[i,j]<=MATCH_UM: gmap[int(pid[i])]=int(gid[j])
    gtset=set((int(u),int(v)) for u,v in gt_edges)
    TP=0; FP=0; cov=set()
    for u,v in edges:
        gu=gmap.get(u); gv=gmap.get(v)
        if gu is None or gv is None: continue      # etiketsiz -> yoksay
        if (gu,gv) in gtset: TP+=1; cov.add((gu,gv))
        else: FP+=1
    FN=len(gtset)-len(cov)
    J=TP/max(TP+FP+FN,1)
    return dict(jaccard=round(J,4), TP=TP, FP=FP, FN=FN,
                node_recall=round(len(set(gmap.values()))/max(len(gt_ndf),1),4),
                pred_nodes=len(nodes))
print("ok")

### 5a · Sanity check: GT → GT skoru **1.0** olmalı

In [ ]:
g_ndf,g_edges=load_geff(TRAIN/(test_names[0]+".geff"))
gt_nodes=[(int(r.id),int(r.t),float(r.z),float(r.y),float(r.x)) for r in g_ndf.itertuples()]
gt_edge_list=[(int(u),int(v)) for u,v in g_edges]
chk=eval_vs_gt(gt_nodes, gt_edge_list, g_ndf, g_edges)
print("GT->GT:", chk)
assert chk["jaccard"]>0.999, "SANITY FAIL — metrik implementasyonu hatali!"
print(">> Metrik implementasyonu dogrulandi.")

## 6 · Baseline'ı değerlendir (test isimli 4 dataset, train GT ile)
Bu 4 film hem train hem test'te → GT'leri elimizde. Baseline **öğrenmeli değil**
(fit edilen parametre yok) → bu değerlendirme **yanlı değil**, gerçek skor tahmini.

In [ ]:
res=[]
for nm in test_names:
    t0=time.time()
    arr=open_image(TEST/(nm+".zarr"))
    nodes,edges=track_dataset(arr)
    gp=TRAIN/(nm+".geff")
    if gp.exists():
        gn,ge=load_geff(gp)
        r=eval_vs_gt(nodes,edges,gn,ge); r["dataset"]=nm; r["sec"]=round(time.time()-t0,1)
        r["gt_nodes"]=len(gn); res.append(r)
        print(nm, r)
    else:
        print(nm, "GT yok (gercek test) — atlandi")
if res:
    df=pd.DataFrame(res)
    print("\n=== OZET ===")
    print(df[["dataset","jaccard","node_recall","TP","FP","FN","pred_nodes","gt_nodes","sec"]].to_string(index=False))
    print("\nORTALAMA Edge Jaccard: %.4f"%df.jaccard.mean())
    print("ORTALAMA node recall : %.4f"%df.node_recall.mean())

## 7 · Gönderim — `test/` dinamik gez, `submission.csv` yaz
Şema: `id,dataset,row_type,node_id,t,z,y,x,source_id,target_id`
- **node** satırı: koordinatlı, `source_id=target_id=-1`
- **edge** satırı: `node_id=t=z=y=x=-1`, `source_id→target_id`

In [ ]:
names = test_names if MAX_TEST is None else test_names[:MAX_TEST]
rows=[]; gid=0; t_start=time.time()
for k,nm in enumerate(names,1):
    t0=time.time()
    arr=open_image(TEST/(nm+".zarr"))
    nodes,edges=track_dataset(arr)
    for nid,t,z,y,x in nodes:
        rows.append((gid,nm,"node",nid,t,z,y,x,-1,-1)); gid+=1      # koordinatlar zaten int
    for u,v in edges:
        rows.append((gid,nm,"edge",-1,-1,-1,-1,-1,u,v)); gid+=1
    if not nodes:   # SART: her test dataseti submission'da yer almali
        rows.append((gid,nm,"node",1,0,0,0,0,-1,-1)); gid+=1
        print(f"  [uyari] {nm}: hic tespit yok -> yer tutucu node eklendi")
    print(f"[{k}/{len(names)}] {nm}: node={len(nodes)} edge={len(edges)} ({time.time()-t0:.0f}s)")
sub=pd.DataFrame(rows, columns=["id","dataset","row_type","node_id","t","z","y","x","source_id","target_id"])
sub.to_csv(OUT_CSV, index=False)
print(f"\nYAZILDI: {OUT_CSV} | satir={len(sub):,} | sure={(time.time()-t_start)/60:.1f} dk")

### 7a · Şema doğrulama

In [ ]:
ss=ROOT/"sample_submission.csv"
if ss.exists():
    s=pd.read_csv(ss)
    print("beklenen kolonlar:", list(s.columns))
    print("bizim kolonlar   :", list(sub.columns))
    assert list(sub.columns)==list(s.columns), "KOLON UYUSMAZLIGI!"
print("\nrow_type:", dict(sub.row_type.value_counts()))
print("dataset :", sub.dataset.nunique(), "adet")
# SART: her test dataseti submission'da olmali
missing=set(names)-set(sub.dataset.unique())
print("eksik dataset:", missing if missing else "yok")
assert not missing, "Bazi test datasetleri submission'da YOK!"
# SART: koordinatlar tamsayi olmali
nd=sub[sub.row_type=="node"]
assert all(nd[c].map(lambda v: float(v).is_integer()).all() for c in ["t","z","y","x"]), "Koordinatlar tamsayi degil!"
print("koordinatlar tamsayi: OK")
print("\nilk node satirlari:"); print(sub[sub.row_type=="node"].head(3).to_string(index=False))
print("\nilk edge satirlari:"); print(sub[sub.row_type=="edge"].head(3).to_string(index=False))
# tutarlilik: her edge'in uclari o dataset'in node_id'lerinde olmali
bad=0
for ds,g in sub.groupby("dataset"):
    nid=set(g[g.row_type=="node"].node_id)
    e=g[g.row_type=="edge"]
    bad+=(~e.source_id.isin(nid)).sum()+(~e.target_id.isin(nid)).sum()
print("\ngecersiz edge referansi:", bad)
assert bad==0, "Edge referans hatasi!"
print(">> Submission gecerli.")

## 8 · Sonraki adımlar

- [ ] **Skoru gör:** submit → public LB (placeholder değil, gerçek gizli test)
- [ ] **Detection tuning:** `FOOT`/`SIGMA`/eşik → node_recall'ı yükselt (kaçırdığımız GT hücre = kayıp edge)
- [ ] **Fazla-tahmin cezası:** `pred_nodes/kare` ~213 civarında mı? Çok yüksekse eşiği sıkılaştır
- [ ] **Bölünme (%10):** şu an yok — Hungarian 1-1. Ebeveyn-kız ~6 µm; ikinci bir eşleştirme turu ekle
- [ ] **Ultrack upgrade:** instance ayrımı zayıfsa (yerel kontrast ~1.5×) çoklu-hipotez segmentasyon
- [ ] **Runtime:** gerçek test seti daha büyük → `detect_frame` hızlandırma gerekebilir